# ML-08 — Capstone Modeling Lane

**Lane:** Structured Content Archetype Clustering  
**Development window:** March 2026  
**Validation design:** client-grouped holdout inside March  
**Sealed final month:** June 2026 — not touched here

This notebook builds the Week-5 clustering model after the earlier framing, data contract, signal audit, and Week-4 rule baseline.

The capstone question is:

> **What performance archetypes exist across the content inventory?**

The Week-4 baseline was a transparent CTR-opportunity rule, not a clustering model. To compare it honestly with clustering, I evaluate both on the **same March validation rows, the same five standardized features, and the same silhouette metric**. This is a narrow structural comparison; it does not claim that silhouette proves business value or that the two methods solve exactly the same decision problem.

No client names, URLs, titles, domains, or raw queries are displayed.


## 1. Method choice and why

I use **K-Means clustering** as the first capstone model.

Why it fits this lane:

- the question asks for natural performance groups rather than a supervised label;
- the feature set is numeric and page-level;
- K-Means is simple enough to inspect and explain;
- I can test a small range of cluster counts without rewarding complexity;
- the output can be checked with cluster profiles, sizes, silhouette score, and low-silhouette examples.

I keep the same five honest March features from the Week-3 data contract:

1. `log_impressions`
2. `ctr_pct`
3. `avg_position`
4. `active_days`
5. `engagement_rate_pct`

The model is decision-support only. A cluster is a measured pattern, not a causal explanation.


In [ ]:
%pip -q install duckdb pandas numpy scikit-learn

import os, json, hashlib
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.getenv("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN not found. Add a Colab Secret named HF_TOKEN.")

con = duckdb.connect()
safe_token = HF_TOKEN.replace("'", "''")
con.execute(f"CREATE OR REPLACE SECRET hf_secret (TYPE huggingface, TOKEN '{safe_token}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"{BASE}/fact_content_daily_performance/month=2026-03/*.parquet"

FEATURES = ["log_impressions","ctr_pct","avg_position","active_days","engagement_rate_pct"]
RANDOM_STATE = 42
OUTPUT_DIR = Path("work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

feature_query = f"""
WITH march AS (
    SELECT
        client_id,
        content_id,
        SUM(COALESCE(gsc_impressions, 0)) AS impressions,
        SUM(COALESCE(gsc_clicks, 0)) AS clicks,
        SUM(CASE
            WHEN gsc_impressions > 0 AND gsc_avg_position > 0
            THEN gsc_avg_position * gsc_impressions ELSE 0 END
        ) AS weighted_position_sum,
        COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS active_days,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN COALESCE(ga4_sessions, 0) ELSE 0 END) AS ga4_sessions_available,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN COALESCE(ga4_engaged_sessions, 0) ELSE 0 END) AS ga4_engaged_sessions_available
    FROM read_parquet('{MARCH}')
    GROUP BY 1, 2
)
SELECT
    client_id,
    content_id,
    LN(1 + impressions) AS log_impressions,
    100.0 * clicks / NULLIF(impressions, 0) AS ctr_pct,
    weighted_position_sum / NULLIF(impressions, 0) AS avg_position,
    active_days,
    100.0 * ga4_engaged_sessions_available / NULLIF(ga4_sessions_available, 0) AS engagement_rate_pct,
    impressions,
    clicks
FROM march
WHERE impressions > 0 AND weighted_position_sum > 0
"""

frame = con.sql(feature_query).df()

def make_page_ref(row):
    raw = f"{row['client_id']}::{row['content_id']}".encode("utf-8")
    return hashlib.sha256(raw).hexdigest()[:10]

frame["page_ref"] = frame.apply(make_page_ref, axis=1)

print(f"March page rows: {len(frame):,}")
print(f"Distinct clients: {frame['client_id'].nunique():,}")
display(frame[FEATURES].describe().T.round(3))


## 2. Split design

I use a **client-grouped holdout**, not a random row split.

Pages from the same client can share site structure, history, and traffic patterns. If one client's pages appear in both train and validation, the result can look cleaner than it really is.

I therefore:

- split March rows by `client_id`;
- use about 75% of clients for fitting;
- use about 25% unseen clients for validation;
- fit imputation and scaling on training rows only;
- fit K-Means on training clients only;
- evaluate K-Means and the Week-4 baseline on the same validation rows.

June remains sealed.


In [ ]:
groups = frame["client_id"]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx, val_idx = next(splitter.split(frame, groups=groups))

train = frame.iloc[train_idx].copy()
val = frame.iloc[val_idx].copy()

train_clients = set(train["client_id"])
val_clients = set(val["client_id"])

assert train_clients.isdisjoint(val_clients), "Client leakage detected."

print(f"Training rows: {len(train):,}")
print(f"Validation rows: {len(val):,}")
print(f"Training clients: {len(train_clients):,}")
print(f"Validation clients: {len(val_clients):,}")
print("PASS: no client appears in both train and validation.")


## 3. Train + compare vs my baseline

I test `k = 2` through `6` and select the strongest validation silhouette score. More clusters are not automatically better.

For the Week-4 comparison, I rebuild its action/no-action partition using **training-derived** position-bucket CTR benchmarks, apply it to the same validation rows, and score that partition with silhouette too.

This gives the required same-data, same-split, same-metric comparison, while keeping the caveat that the baseline was originally a review queue rather than a clustering method.


In [ ]:
preprocess = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

X_train = preprocess.fit_transform(train[FEATURES])
X_val = preprocess.transform(val[FEATURES])

candidate_rows = []
models = {}

for k in range(2, 7):
    model = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE)
    train_labels = model.fit_predict(X_train)
    val_labels = model.predict(X_val)

    candidate_rows.append({
        "k": k,
        "train_silhouette": silhouette_score(X_train, train_labels),
        "validation_silhouette": silhouette_score(X_val, val_labels),
        "smallest_validation_cluster_n": int(pd.Series(val_labels).value_counts().min()),
    })
    models[k] = model

k_table = pd.DataFrame(candidate_rows).sort_values("validation_silhouette", ascending=False).reset_index(drop=True)
display(k_table.round(3))

best_k = int(k_table.loc[0, "k"])
best_model = models[best_k]
best_val_labels = best_model.predict(X_val)
model_silhouette = silhouette_score(X_val, best_val_labels)

print(f"Selected k = {best_k}")
print(f"Validation silhouette = {model_silhouette:.3f}")


In [ ]:
MIN_IMPRESSIONS = 100
MIN_ACTIVE_DAYS = 7
MAX_POSITION = 20.0

def add_position_bucket(df):
    out = df.copy()
    out["position_bucket"] = pd.cut(
        out["avg_position"],
        bins=[0, 3, 10, 20],
        labels=["top_3", "page_1_rest", "page_2"],
        include_lowest=True,
    )
    return out

train_b = add_position_bucket(train)

eligible_train = train_b.loc[
    train_b["impressions"].ge(MIN_IMPRESSIONS)
    & train_b["active_days"].ge(MIN_ACTIVE_DAYS)
    & train_b["avg_position"].gt(0)
    & train_b["avg_position"].le(MAX_POSITION)
    & train_b["ctr_pct"].notna()
].copy()

benchmarks = (
    eligible_train.groupby("position_bucket", observed=False)
    .agg(bucket_clicks=("clicks","sum"), bucket_impressions=("impressions","sum"))
)
benchmarks["expected_ctr_pct"] = (
    100.0 * benchmarks["bucket_clicks"] / benchmarks["bucket_impressions"].replace(0, np.nan)
)

val_b = add_position_bucket(val)
val_b = val_b.join(benchmarks["expected_ctr_pct"], on="position_bucket")

baseline_action = (
    val_b["impressions"].ge(MIN_IMPRESSIONS)
    & val_b["active_days"].ge(MIN_ACTIVE_DAYS)
    & val_b["avg_position"].gt(0)
    & val_b["avg_position"].le(MAX_POSITION)
    & val_b["ctr_pct"].notna()
    & val_b["expected_ctr_pct"].notna()
    & val_b["ctr_pct"].lt(val_b["expected_ctr_pct"])
).astype(int)

counts = baseline_action.value_counts().sort_index()
display(counts.rename_axis("baseline_partition").reset_index(name="n"))

if baseline_action.nunique() >= 2 and counts.min() >= 2:
    baseline_silhouette = silhouette_score(X_val, baseline_action)
else:
    baseline_silhouette = np.nan

comparison = pd.DataFrame({
    "method": ["Week-4 rule partition", f"K-Means (k={best_k})"],
    "validation_rows": [len(val), len(val)],
    "metric": ["silhouette", "silhouette"],
    "validation_score": [baseline_silhouette, model_silhouette],
    "same_validation_split": [True, True],
    "future_or_label_inputs": [False, False],
})

display(comparison.round(3))

if np.isfinite(baseline_silhouette):
    print(f"K-Means minus baseline silhouette: {model_silhouette - baseline_silhouette:+.3f}")
else:
    print("Baseline silhouette undefined because the validation partition did not form two usable groups.")


## 4. Errors and interpretation

There is no supervised ground-truth label, so I do not describe a page as “wrong” in the usual classification sense.

Instead I inspect:

1. cluster profiles in the original units;
2. cluster sizes;
3. per-page silhouette values.

Low or negative silhouette values are my practical error cases because those pages fit their assigned cluster poorly or sit near another cluster boundary.

I treat the patterns as descriptive, not causal.


In [ ]:
val_result = val.copy()
val_result["cluster"] = best_val_labels

profile = (
    val_result.groupby("cluster")
    .agg(
        n=("page_ref","size"),
        log_impressions_median=("log_impressions","median"),
        ctr_pct_median=("ctr_pct","median"),
        avg_position_median=("avg_position","median"),
        active_days_median=("active_days","median"),
        engagement_rate_pct_median=("engagement_rate_pct","median"),
    )
    .sort_index()
)
display(profile.round(3))

sample_sil = silhouette_samples(X_val, best_val_labels)
val_result["sample_silhouette"] = sample_sil

error_summary = pd.DataFrame({
    "diagnostic": [
        "validation silhouette",
        "median sample silhouette",
        "share sample silhouette < 0",
        "share sample silhouette < 0.10",
    ],
    "value": [
        model_silhouette,
        float(np.median(sample_sil)),
        float(np.mean(sample_sil < 0)),
        float(np.mean(sample_sil < 0.10)),
    ],
})
display(error_summary.round(3))

ambiguous = (
    val_result.sort_values("sample_silhouette")
    .head(10)[["page_ref","cluster","sample_silhouette"] + FEATURES]
)
print("Ten weakest-fitting validation pages:")
display(ambiguous.round(3))


In [ ]:
centers_std = pd.DataFrame(best_model.cluster_centers_, columns=FEATURES)
centers_std.index.name = "cluster"

print("Standardized cluster centers:")
display(centers_std.round(3))

feature_spread = (centers_std.max(axis=0) - centers_std.min(axis=0)).sort_values(ascending=False)
feature_spread_table = feature_spread.rename("standardized_center_range").reset_index().rename(columns={"index":"feature"})

print("Features that separate cluster centers most:")
display(feature_spread_table.round(3))

print("This is descriptive separation, not causal feature importance.")


In [ ]:
metrics = {
    "lane": "structured_content_archetype_clustering",
    "development_window": "2026-03",
    "split_design": "client_grouped_holdout",
    "train_rows": int(len(train)),
    "validation_rows": int(len(val)),
    "train_clients": int(len(train_clients)),
    "validation_clients": int(len(val_clients)),
    "features": FEATURES,
    "candidate_k": list(range(2,7)),
    "selected_k": int(best_k),
    "model_validation_silhouette": float(model_silhouette),
    "baseline_validation_silhouette": None if not np.isfinite(baseline_silhouette) else float(baseline_silhouette),
    "negative_sample_silhouette_share": float(np.mean(sample_sil < 0)),
    "low_sample_silhouette_share_lt_0_10": float(np.mean(sample_sil < 0.10)),
    "june_sealed": True,
    "future_or_label_inputs_used": False,
}

metrics_path = OUTPUT_DIR / "w05_model_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

print(json.dumps(metrics, indent=2))
print(f"Wrote metrics receipt to: {metrics_path}")


## Self-check

Before you submit, confirm each line honestly:

- [x] Method choice matches the Structured Content Archetype Clustering lane.
- [x] March-only honest features are used.
- [x] June remains sealed.
- [x] Split is grouped by client.
- [x] Imputation and scaling are fitted on training clients only.
- [x] K-Means is fitted on training clients only.
- [x] `k` is chosen from a small explicit candidate range.
- [x] Week-4 baseline and K-Means use the same validation rows.
- [x] The comparison uses the same metric: silhouette.
- [x] The notebook explains the limitation of comparing a rule partition with clustering.
- [x] Cluster profiles and low-silhouette cases are inspected.
- [x] No future-window or label-derived inputs are used.
- [x] No client names, URLs, titles, domains, or raw queries are displayed.
- [x] A metrics JSON is written to `work/outputs/w05_model_metrics.json`.
- [ ] Run all cells in Colab with warehouse access and confirm the real outputs are visible.
- [ ] Read the Week-5 research paper linked on the assignment card.
- [ ] Save as `work/notebooks/w05_model.ipynb`.
- [ ] Commit the executed notebook and metrics JSON.
- [ ] Submit the repo URL.
